In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm
from torch.utils.data import DataLoader, Subset
from bioplnn.models import SpatiallyEmbeddedClassifier, SpatiallyEmbeddedRNN
from bioplnn.utils import initialize_dataloader
from collections import deque
import pickle
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 300

checkpoint_path = "./train/checkpoints/"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
maze_data_path = "./data/mazes/"

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, in_channels=4, num_classes=2, dropout=0.3):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=5, padding=2)
        self.conv2 = nn.Conv2d(64, 64, kernel_size=5, padding=2)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=5, padding=2)
        self.pool  = nn.MaxPool2d(2, 2)
        self.relu  = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(dropout)
        self.gap   = nn.AdaptiveAvgPool2d(1)
        self.fc1   = nn.Linear(128, 512)
        self.fc2   = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.pool(self.relu(self.conv3(x)))
        x = self.gap(x)
        x = torch.flatten(x, 1)
        x = self.dropout(self.relu(self.fc1(x)))
        return self.fc2(x)


def prepare_rnn_weights(state_dict):
    new_state_dict = {}
    for key, value in state_dict.items():
        if key.startswith('rnn.'):
            new_state_dict[key[4:]] = value
        elif not key.startswith('readout'):
            new_state_dict[key] = value
    return new_state_dict


def load_model_and_config(wandb_name, checkpoint_path):
    try:
        full_cfg = pickle.load(open(checkpoint_path + f"{wandb_name}.pkl", "rb"))
        model_cfg, num_steps = full_cfg["model_config"], full_cfg["num_steps"]
    except:
        model_cfg = pickle.load(open(checkpoint_path + f"{wandb_name}.pkl", "rb"))
        num_steps = 20

    try:
        if full_cfg["model_type"] == "cnn":
            model = SimpleCNN(in_channels=model_cfg["in_channels"], num_classes=model_cfg["num_classes"])
            classifier = SimpleCNN(in_channels=model_cfg["in_channels"], num_classes=model_cfg["num_classes"])
        else:
            model = SpatiallyEmbeddedRNN(**model_cfg["rnn_kwargs"])
            classifier = SpatiallyEmbeddedClassifier(**model_cfg)
    except:
        model = SpatiallyEmbeddedRNN(**model_cfg["rnn_kwargs"])
        classifier = SpatiallyEmbeddedClassifier(**model_cfg)

    try:
        state_dict = torch.load(checkpoint_path + f"{wandb_name}_{checkpoint}.pth", map_location=torch.device('cpu'))
    except:
        state_dict = torch.load(checkpoint_path + f"{wandb_name}/{checkpoint}.pth", map_location=torch.device('cpu'))

    try:
        state_dict = state_dict["model_state"]
    except:
        pass

    model.load_state_dict(prepare_rnn_weights(state_dict))
    classifier.load_state_dict(state_dict)

    return model, classifier, num_steps, full_cfg, state_dict

In [ ]:
def get_decisions(classifier, dataloader, decision_step, device=None):
    if device is None:
        device = next(classifier.parameters()).device

    num_samples = len(dataloader.dataset)
    decisions = np.zeros((num_samples,), dtype=np.int64)

    classifier.eval()
    classifier.to(device)

    sample_idx = 0
    with torch.no_grad():
        for batch_inputs, _labels in tqdm(dataloader):
            bsz = batch_inputs.shape[0]
            batch_inputs = batch_inputs.to(device, non_blocking=True)
            pred = classifier(batch_inputs, num_steps=decision_step)
            dec = torch.argmax(pred, dim=-1).detach().cpu().numpy()
            decisions[sample_idx:sample_idx + bsz] = dec
            sample_idx += bsz

    return decisions

In [ ]:
def compute_shortest_path_mask(maze, start_goal):
    """
    BFS shortest path between two single-pixel endpoints.
    maze: 2D array (0=wall, 1=corridor)
    start_goal: 2D array with exactly two nonzero pixels
    Returns: 2D mask with 1s along shortest path (sum = geodesic distance).
    """
    maze = (maze != 0).astype(np.uint8)
    sg = (start_goal != 0).astype(np.uint8)

    pts = np.argwhere(sg == 1)
    if pts.shape[0] != 2:
        return start_goal
    s = tuple(pts[0])
    t = tuple(pts[1])

    maze[s] = 1
    maze[t] = 1

    H, W = maze.shape
    visited = np.zeros_like(maze, dtype=bool)
    parent = -np.ones((H, W, 2), dtype=np.int32)

    nbrs = [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]
    q = deque([s])
    visited[s] = True

    found = False
    while q:
        r, c = q.popleft()
        if (r, c) == t:
            found = True
            break
        for dr, dc in nbrs:
            nr, nc = r + dr, c + dc
            if not (0 <= nr < H and 0 <= nc < W):
                continue
            if visited[nr, nc] or maze[nr, nc] != 1:
                continue
            # Block diagonal if both orthogonal neighbors are walls
            if dr != 0 and dc != 0:
                if maze[r + dr, c] == 0 and maze[r, c + dc] == 0:
                    continue
            visited[nr, nc] = True
            parent[nr, nc] = (r, c)
            q.append((nr, nc))

    path_map = np.zeros_like(maze, dtype=np.uint8)
    if not found:
        return path_map

    cur = t
    while True:
        path_map[cur] = 1
        if cur == s:
            break
        pr, pc = parent[cur]
        if pr == -1:
            path_map[:] = 0
            break
        cur = (int(pr), int(pc))

    return path_map.astype(np.uint8)


def collect_shortest_paths(dataloader):
    """Geodesic distance (shortest path length) for each sample."""
    shortest_path_lengths = []
    with torch.no_grad():
        for batch_inputs, _labels in tqdm(dataloader):
            for inp in batch_inputs:
                inp = np.array(inp)
                maze = inp[0]
                start_goal = inp[-1]

                # Downsample: 2x2 cue blocks -> single pixels
                maze = (maze == 1)[::2, ::2]
                start_goal = (start_goal != 0)[::2, ::2]

                shortest_path = compute_shortest_path_mask(maze, start_goal)
                shortest_path_lengths.append(np.sum(shortest_path))

    return np.array(shortest_path_lengths)


def collect_euclidean_distances(dataloader):
    """Euclidean distance between start and goal for each sample."""
    distances = []
    with torch.no_grad():
        for batch_inputs, _labels in tqdm(dataloader):
            for inp in batch_inputs:
                start_goal = inp[-1]

                # Downsample: 2x2 cue blocks -> single pixels
                sg = (start_goal != 0)[::2, ::2]
                rows, cols = np.where(sg)
                pts = np.column_stack((rows, cols))

                if len(pts) != 2:
                    distance = 0
                else:
                    distance = np.sqrt((pts[0][0] - pts[1][0])**2 + (pts[0][1] - pts[1][1])**2)
                distances.append(distance)
    return distances

## Load model

In [ ]:
wandb_name, checkpoint = "rosy-morning-40", 390

model, classifier, num_steps, cfg, state_dict = load_model_and_config(wandb_name, checkpoint_path)
model.eval()
model.to(device)
classifier.eval()
classifier.to(device);

## Load data

In [ ]:
loader, test_loader = initialize_dataloader(
    seed=42, root=maze_data_path, batch_size=256, dataset="mazes", annotation_scaling=3
)

num_samples = 10000
all_indices = np.arange(len(loader.dataset))
subset_indices = np.random.choice(all_indices, size=num_samples, replace=False)
decision_step = 20

subset_dataset = Subset(loader.dataset, subset_indices)
loader = DataLoader(
    subset_dataset,
    batch_size=256,
    shuffle=False,
    drop_last=False
)

## Compute distances and decision points

In [ ]:
shortest_path_lengths = collect_shortest_paths(loader)
euclidean_distances = collect_euclidean_distances(loader)

final_decisions = get_decisions(classifier, loader, decision_step)
decision_points = np.zeros(len(loader.dataset)) + 20
for step in range(20, 0, -1):
    decisions = get_decisions(classifier, loader, step)
    for input_idx, decision in enumerate(decisions):
        if decision == final_decisions[input_idx]:
            decision_points[input_idx] = step

In [ ]:
mask = (final_decisions > 0) & (shortest_path_lengths > 4)

## Time to solve vs Euclidean distance

In [ ]:
plt.scatter(np.array(euclidean_distances)[mask], decision_points[mask])
plt.title("Time to Decision vs Euclidean Distance", fontsize=14)
plt.ylabel("Time to Decision", fontsize=12)
plt.xlabel("Euclidean Distance Between Start and Goal", fontsize=12)
plt.plot(np.unique(np.array(euclidean_distances)[mask]), np.poly1d(np.polyfit(np.array(euclidean_distances)[mask], decision_points[mask], 1))(np.unique(np.array(euclidean_distances)[mask])), color='red')
r_squared = np.corrcoef(np.array(euclidean_distances)[mask], decision_points[mask])[0,1]**2
plt.legend([f"R-squared: {r_squared:.2f}"], loc="best", fontsize=12)
plt.show()

## Time to solve vs Geodesic distance (shortest path)

In [ ]:
plt.scatter(shortest_path_lengths[mask], decision_points[mask])
plt.title("Time to Decision vs Geodesic Distance", fontsize=14)
plt.ylabel("Time to Decision", fontsize=12)
plt.xlabel("Geodesic Distance Between Start and Goal", fontsize=12)
plt.plot(np.unique(np.array(shortest_path_lengths)[mask]), np.poly1d(np.polyfit(np.array(shortest_path_lengths)[mask], decision_points[mask], 1))(np.unique(np.array(shortest_path_lengths)[mask])), color='red')
r_squared = np.corrcoef(np.array(shortest_path_lengths)[mask], decision_points[mask])[0,1]**2
plt.legend([f"R-squared: {r_squared:.2f}"], loc="best", fontsize=12)
plt.show()